# S13.2 - LoRA Financial Instruction Tuning

This notebook trains a LoRA adapter for a small causal language model. Instead of adding a classification head, we format the Apple 10-K sentence labels as instruction-response examples so students can see how supervised fine-tuning teaches a stable output format.

Colab setup:

```bash
pip install -q transformers datasets accelerate peft
```

In [1]:
from pathlib import Path
import sys

def find_finance_dir(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for path in [start, *start.parents]:
        candidate = path / "help-code" / "4-nlp-finance"
        if candidate.exists():
            return candidate
        if path.name == "4-nlp-finance":
            return path
    raise FileNotFoundError("Run this notebook from the course repo or help-code/4-nlp-finance.")

FINANCE_DIR = find_finance_dir()
SRC_DIR = FINANCE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from finance_nlp_utils import load_evaluation_csv

rows = load_evaluation_csv("apple_2025_sentence_classification.csv")
labels = sorted({row["label"] for row in rows})
labels

['cash_capital',
 'legal_regulatory',
 'margin_cost',
 'operations_supply',
 'other_disclosure',
 'revenue_sales',
 'risk_factor']

In [2]:
def format_example(sentence: str, label: str | None = None) -> str:
    label_text = "" if label is None else label
    return f"""### Instruction:
Classify the Apple 10-K disclosure sentence into one of these labels:
{', '.join(labels)}.

### Input:
{sentence}

### Response:
{label_text}"""

print(format_example(rows[0]["sentence"], rows[0]["label"]))

### Instruction:
Classify the Apple 10-K disclosure sentence into one of these labels:
cash_capital, legal_regulatory, margin_cost, operations_supply, other_disclosure, revenue_sales, risk_factor.

### Input:
Business Company Background The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services.

### Response:
revenue_sales


In [3]:
from datasets import Dataset, DatasetDict

splits = {}
for split in ["train", "validation", "test"]:
    examples = [
        {"text": format_example(row["sentence"], row["label"]), "sentence": row["sentence"], "label": row["label"]}
        for row in rows
        if row["split"] == split
    ]
    splits[split] = Dataset.from_list(examples)

dataset = DatasetDict(splits)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'sentence', 'label'],
        num_rows: 112
    })
    validation: Dataset({
        features: ['text', 'sentence', 'label'],
        num_rows: 16
    })
    test: Dataset({
        features: ['text', 'sentence', 'label'],
        num_rows: 32
    })
})

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

c:\Program Files\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nakul\.cache\huggingface\hub\models--sshleifer--tiny-gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 128 || all params: 102,842 || trainable%: 0.1245


c:\Program Files\Python312\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [5]:
def tokenize(batch):
    tokens = tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset["train"].column_names)
tokenized.set_format("torch")
tokenized

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 112
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 16
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 32
    })
})

In [6]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=str(FINANCE_DIR / "outputs" / "sec-risk-lora"),
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
)
trainer.train()
model.save_pretrained(FINANCE_DIR / "outputs" / "sec-risk-lora-adapter")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,10.710971,10.710557
2,10.709544,10.710533
3,10.710220,10.710526


In [7]:
import torch

prompt = format_example("The Company depends on third-party suppliers for components.", None)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(generated[0], skip_special_tokens=True))

### Instruction:
Classify the Apple 10-K disclosure sentence into one of these labels:
cash_capital, legal_regulatory, margin_cost, operations_supply, other_disclosure, revenue_sales, risk_factor.

### Input:
The Company depends on third-party suppliers for components.

### Response:
 factors factors factors factors factors factors factors factors
